# Karting notebook amp-up

September 21, 2025

Analysis of my Karting experiences!

## Make the json number falues actually numbers

In [ ]:
import orjson
import re

def is_numeric_string(value):
    """Check if a string represents a number (int or float)"""
    if not isinstance(value, str):
        return False
    
    # Remove whitespace
    value = value.strip()
    
    # Check if it matches a number pattern (including negative numbers and decimals)
    number_pattern = r'^-?(?:\d+\.?\d*|\.\d+)$'
    return bool(re.match(number_pattern, value))

def convert_string_to_number(value):
    """Convert a numeric string to int or float"""
    if '.' in value:
        return float(value)
    else:
        return int(value)

def convert_json_strings_to_numbers(data):
    """Recursively convert string values that represent numbers to actual numbers"""
    if isinstance(data, dict):
        return {key: convert_json_strings_to_numbers(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [convert_json_strings_to_numbers(item) for item in data]
    elif isinstance(data, str) and is_numeric_string(data):
        return convert_string_to_number(data)
    else:
        return data

def process_json_file(json_path, output_path=None):
    """Read JSON file, convert string numbers, and optionally save to new file"""
    try:
        # Read the JSON file
        with open(json_path, 'rb') as file:
            data = orjson.loads(file.read())
        
        print(f"Original data type: {type(data)}")
        if isinstance(data, dict):
            print(f"Original data has {len(data)} keys")
        elif isinstance(data, list):
            print(f"Original data has {len(data)} items")
        
        # Convert string numbers to actual numbers
        converted_data = convert_json_strings_to_numbers(data)
        
        # Show some examples of conversions
        print("\nExample conversions:")
        if isinstance(converted_data, dict):
            for i, (key, value) in enumerate(converted_data.items()):
                if i < 5:  # Show first 5 items
                    original_value = data[key] if isinstance(data, dict) else None
                    if original_value != value:
                        print(f"  {key}: '{original_value}' -> {value} ({type(value).__name__})")
                if i >= 10:  # Limit output
                    break
        
        # Save to output file if specified
        if output_path:
            with open(output_path, 'wb') as file:
                file.write(orjson.dumps(converted_data, option=orjson.OPT_INDENT_2))
            print(f"\nConverted data saved to: {output_path}")
        
        return converted_data
        
    except FileNotFoundError:
        print(f"Error: File not found at {json_path}")
        return None
    except orjson.JSONDecodeError as e:
        print(f"Error: Invalid JSON format - {e}")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

# Main execution
json_path = "/Users/nathanverrill/cota-karting/2025-09-21_16_48_05_nathan-iphone.json"

# Process the file
converted_data = process_json_file(json_path)

if converted_data:
    print(f"\nConversion completed successfully!")
    
    # Optionally save to a new file
    save_converted = input("Would you like to save the converted data to a new file? (y/n): ").lower().strip()
    if save_converted == 'y':
        output_path = json_path.replace('.json', '_converted.json')
        with open(output_path, 'wb') as file:
            file.write(orjson.dumps(converted_data, option=orjson.OPT_INDENT_2))
        print(f"Converted data saved to: {output_path}")
    
    # Show a sample of the converted data
    print("\nSample of converted data:")
    if isinstance(converted_data, dict):
        for i, (key, value) in enumerate(converted_data.items()):
            if i < 10:  # Show first 10 items
                print(f"  {key}: {value} ({type(value).__name__})")
            else:
                print("  ...")
                break
    elif isinstance(converted_data, list) and len(converted_data) > 0:
        print(f"  First item: {converted_data[0]}")
        print(f"  Type: {type(converted_data[0])}")

## Load into dataframe with timestamp datatype

In [ ]:
import orjson
import polars as pl
from datetime import datetime
import re

def load_json_to_polars(json_path):
    """Load JSON data into Polars DataFrame with proper timestamp handling"""
    
    # Read the converted JSON file
    with open(json_path, 'rb') as file:
        data = orjson.loads(file.read())
    
    # If it's a single object, wrap it in a list
    if isinstance(data, dict):
        data = [data]
    
    print(f"Loaded {len(data)} records")
    
    # Create the initial DataFrame
    df = pl.DataFrame(data)
    
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    
    # Identify timestamp columns and their types
    timestamp_columns = {}
    
    for col in df.columns:
        # Check for different timestamp patterns
        if any(pattern in col.lower() for pattern in ['timestamp', 'date', 'time']):
            sample_value = df[col].drop_nulls().first()
            
            if sample_value is not None:
                if isinstance(sample_value, str):
                    # ISO datetime strings (e.g., "2025-09-18T17:52:17.207-05:00")
                    if re.match(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', str(sample_value)):
                        timestamp_columns[col] = 'iso_datetime'
                elif isinstance(sample_value, (int, float)):
                    # Unix timestamps (seconds since epoch)
                    if 'since1970' in col or 'Timestamp_since1970' in col:
                        timestamp_columns[col] = 'unix_seconds'
                    # Timestamps since reboot (relative timestamps in seconds)
                    elif 'sinceReboot' in col or 'sinceReboot' in col:
                        timestamp_columns[col] = 'relative_seconds'
    
    print(f"\nIdentified timestamp columns:")
    for col, ts_type in timestamp_columns.items():
        print(f"  {col}: {ts_type}")
    
    # Convert timestamp columns to appropriate types
    for col, ts_type in timestamp_columns.items():
        try:
            if ts_type == 'iso_datetime':
                # Convert ISO datetime strings to datetime
                df = df.with_columns([
                    pl.col(col).str.to_datetime().alias(col)
                ])
                print(f"✓ Converted {col} to datetime")
                
            elif ts_type == 'unix_seconds':
                # Convert Unix timestamp (seconds since 1970) to datetime
                df = df.with_columns([
                    pl.from_epoch(pl.col(col), time_unit="s").alias(col)
                ])
                print(f"✓ Converted {col} from Unix timestamp to datetime")
                
            elif ts_type == 'relative_seconds':
                # Keep relative timestamps as float (seconds since reboot)
                # These are relative times, so we'll keep them as duration/float
                df = df.with_columns([
                    pl.col(col).cast(pl.Float64).alias(col)
                ])
                print(f"✓ Ensured {col} is Float64 (relative timestamp)")
                
        except Exception as e:
            print(f"⚠ Warning: Could not convert {col}: {e}")
    
    # Show data types
    print(f"\nDataFrame dtypes:")
    for col, dtype in zip(df.columns, df.dtypes):
        if col in timestamp_columns:
            print(f"  {col}: {dtype} ⏰")
        else:
            print(f"  {col}: {dtype}")
    
    return df

def analyze_timestamps(df):
    """Analyze the timestamp data for insights"""
    print(f"\n" + "="*50)
    print("TIMESTAMP ANALYSIS")
    print("="*50)
    
    # Find datetime columns
    datetime_cols = [col for col, dtype in zip(df.columns, df.dtypes) 
                    if dtype == pl.Datetime or 'date' in col.lower() or 'time' in col.lower()]
    
    for col in datetime_cols:
        if df[col].dtype == pl.Datetime:
            min_time = df[col].min()
            max_time = df[col].max()
            print(f"\n{col}:")
            print(f"  Range: {min_time} to {max_time}")
            if min_time and max_time:
                duration = max_time - min_time
                print(f"  Duration: {duration}")
        elif 'since' in col.lower():
            min_val = df[col].min()
            max_val = df[col].max()
            print(f"\n{col}:")
            print(f"  Range: {min_val} to {max_val} seconds")
            if min_val is not None and max_val is not None:
                duration = max_val - min_val
                print(f"  Duration: {duration:.3f} seconds")

# Jupyter Notebook Execution
# Set your file path
json_path = "/Users/nathanverrill/cota-karting/2025-09-21_16_48_05_nathan-iphone_converted.json"

# Load the data
print("Loading JSON data into Polars DataFrame...")
df = load_json_to_polars(json_path)

# Analyze timestamps
analyze_timestamps(df)

# Show sample of the data
print(f"\n" + "="*50)
print("SAMPLE DATA")
print("="*50)
print(df.head(3))

# Show data info
print(f"\n" + "="*50)
print("DATAFRAME INFO")
print("="*50)
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.estimated_size('mb'):.2f} MB")

print(f"\nDataFrame loaded successfully! Use 'df' to access your data.")

## Trace with Sensor Fusion

Using multiple types of data to create a trace, which is more accurate than GPS alone.

In [ ]:
# This script uses a Kalman filter to generate a smooth vehicle trace from
# raw telemetry data, using the Polars library for efficient data manipulation.

import polars as pl
import numpy as np
from typing import Dict, Any

# Define the Kalman Filter class to handle state prediction and correction.
class KalmanFilter:
    """
    A simple Kalman filter implementation for 2D position and velocity.
    """
    def __init__(self, dt: float, initial_state: np.ndarray, process_noise_std: float, measurement_noise_std: float):
        """
        Initializes the Kalman Filter.

        Args:
            dt (float): The time step between measurements.
            initial_state (np.ndarray): The initial state vector [x, y, vx, vy].
            process_noise_std (float): Standard deviation of the process noise.
            measurement_noise_std (float): Standard deviation of the measurement noise.
        """
        # State transition matrix (A) - defines how the state evolves over time
        # [x, y, vx, vy]' = A * [x, y, vx, vy]
        self.A = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], dtype=np.float64)

        # Process noise covariance matrix (Q) - uncertainty in the state prediction
        # We assume independent noise on each state variable.
        self.Q = np.diag([
            0.5 * dt**2 * process_noise_std,
            0.5 * dt**2 * process_noise_std,
            dt * process_noise_std,
            dt * process_noise_std
        ])

        # Measurement matrix (H) - maps the state to the measurements
        # measurements [x, y]' = H * [x, y, vx, vy]'
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ], dtype=np.float64)

        # Measurement noise covariance matrix (R) - uncertainty in the measurements
        self.R = np.diag([measurement_noise_std**2, measurement_noise_std**2])

        # Initial state estimate (x_hat) and covariance (P)
        self.x_hat = initial_state
        self.P = np.diag([1e-2, 1e-2, 1e-2, 1e-2])  # Initial covariance, starts small

    def predict(self) -> None:
        """
        Predicts the next state and the new state covariance.
        """
        self.x_hat = self.A @ self.x_hat
        self.P = self.A @ self.P @ self.A.T + self.Q

    def update(self, measurement: np.ndarray) -> None:
        """
        Updates the state estimate with a new measurement.

        Args:
            measurement (np.ndarray): The new measurement [x_gps, y_gps].
        """
        # Calculate the Kalman Gain
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        # Update the state estimate and covariance
        y = measurement - (self.H @ self.x_hat)
        self.x_hat = self.x_hat + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

    def get_state(self) -> np.ndarray:
        """
        Returns the current estimated state.
        """
        return self.x_hat

def convert_lat_lon_to_xy(lat: float, lon: float, lat_ref: float, lon_ref: float) -> tuple[float, float]:
    """
    Converts latitude and longitude to a simple 2D X-Y coordinate system.

    This is an approximation suitable for small tracks where the Earth's curvature
    can be ignored.
    """
    R_earth = 6371e3  # Radius of Earth in meters
    x = R_earth * np.cos(np.radians(lat_ref)) * np.radians(lon - lon_ref)
    y = R_earth * np.radians(lat - lat_ref)
    return x, y

def process_telemetry_with_kalman(df: pl.DataFrame) -> pl.DataFrame:
    """
    Processes a Polars DataFrame with raw telemetry data to create a
    smoothed trace using a Kalman filter.

    Args:
        df (pl.DataFrame): A DataFrame with raw telemetry data including
                          'locationLatitude', 'locationLongitude', and timestamps.

    Returns:
        pl.DataFrame: A new DataFrame with the smoothed trace in X-Y coordinates.
    """
    if df.is_empty():
        return pl.DataFrame({"trace_x": [], "trace_y": []})

    # --- Step 1: Data Preparation ---
    print("Preparing data...")
    # Sort data by time to ensure chronological processing
    df = df.sort("locationTimestamp_since1970")

    # Handle missing timestamps by computing time differences and converting to seconds
    df = df.with_columns([
        # Compute time differences in seconds, fill nulls with 0.1 seconds
        (pl.col("locationTimestamp_since1970").diff().dt.total_seconds().fill_null(0.1)).alias("dt")
    ])
    
    # Convert seconds to a simple numeric time scale if needed
    df = df.with_columns(
        (pl.col("locationTimestamp_since1970") - pl.col("locationTimestamp_since1970").min()).dt.total_seconds().alias("time_s")
    )

    # Convert lat/lon to a local X-Y coordinate system (in meters)
    lat_ref, lon_ref = df.select(pl.col("locationLatitude").first(), pl.col("locationLongitude").first()).row(0)
    
    # Use map_elements instead of apply for newer Polars versions
    df = df.with_columns([
        pl.struct(["locationLatitude", "locationLongitude"])
        .map_elements(lambda s: convert_lat_lon_to_xy(s["locationLatitude"], s["locationLongitude"], lat_ref, lon_ref), return_dtype=pl.Object)
        .alias("xy_coords")
    ])
    
    df = df.with_columns([
        pl.col("xy_coords").map_elements(lambda c: c[0], return_dtype=pl.Float64).alias("x_gps"),
        pl.col("xy_coords").map_elements(lambda c: c[1], return_dtype=pl.Float64).alias("y_gps")
    ])

    # --- Step 2: Kalman Filtering Loop ---
    print("Applying Kalman filter...")
    kalman_filter_states = []
    
    # Initialize the Kalman filter
    initial_x = df.select(pl.col("x_gps").first()).item()
    initial_y = df.select(pl.col("y_gps").first()).item()
    # Initial speed can be derived from the first two points
    initial_state = np.array([initial_x, initial_y, 0.0, 0.0])
    
    # These noise values are a starting point and would need to be tuned
    # based on the specific sensor and environment characteristics.
    process_noise_std = 0.5   # Process model uncertainty
    measurement_noise_std = 3.0 # GPS measurement uncertainty (in meters)
    
    # Use the first dt value for initialization
    initial_dt = df.select(pl.col("dt").first()).item()
    
    # Create the filter instance
    kf = KalmanFilter(
        dt=initial_dt, 
        initial_state=initial_state, 
        process_noise_std=process_noise_std, 
        measurement_noise_std=measurement_noise_std
    )
    
    for i in range(df.height):
        row = df.row(i, named=True)
        
        # Update the state transition matrix with the current time step
        kf.A = np.array([
            [1, 0, row['dt'], 0],
            [0, 1, 0, row['dt']],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], dtype=np.float64)
        
        # Update the process noise matrix with the current time step
        kf.Q = np.diag([
            0.5 * row['dt']**2 * process_noise_std,
            0.5 * row['dt']**2 * process_noise_std,
            row['dt'] * process_noise_std,
            row['dt'] * process_noise_std
        ])
        
        # Predict the next state
        kf.predict()
        
        # Update with the new GPS measurement
        measurement = np.array([row['x_gps'], row['y_gps']])
        kf.update(measurement)
        
        # Store the corrected state
        kalman_filter_states.append(kf.get_state())

    # --- Step 3: Create the Output DataFrame ---
    print("Creating output DataFrame...")
    
    states_array = np.array(kalman_filter_states)
    
    # Create a new DataFrame with the smoothed trace
    output_df = pl.DataFrame({
        "time_s": df.get_column("time_s"),
        "raw_x": df.get_column("x_gps"),
        "raw_y": df.get_column("y_gps"),
        "trace_x": states_array[:, 0],
        "trace_y": states_array[:, 1],
        "trace_vx": states_array[:, 2],
        "trace_vy": states_array[:, 3]
    })
    
    return output_df

## load and smooth the location data with kalman

In [ ]:
df_smoothed = process_telemetry_with_kalman(df)

In [ ]:
df_smoothed.head()

## draw it

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

def plot_kalman_trace(df_smoothed: pl.DataFrame, title: str = "Vehicle Trace - Raw vs Kalman Filtered"):
    """
    Plot the raw GPS trace vs the Kalman filtered trace.
    
    Args:
        df_smoothed: DataFrame from process_telemetry_with_kalman function
        title: Plot title
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Convert to numpy for plotting
    raw_x = df_smoothed.get_column("raw_x").to_numpy()
    raw_y = df_smoothed.get_column("raw_y").to_numpy()
    trace_x = df_smoothed.get_column("trace_x").to_numpy()
    trace_y = df_smoothed.get_column("trace_y").to_numpy()
    time_s = df_smoothed.get_column("time_s").to_numpy()
    
    # Plot 1: X-Y trajectory comparison
    ax1.scatter(raw_x, raw_y, c='red', alpha=0.6, s=20, label='Raw GPS', marker='o')
    ax1.plot(trace_x, trace_y, 'b-', linewidth=2, label='Kalman Filtered', alpha=0.8)
    ax1.set_xlabel('X Position (meters)')
    ax1.set_ylabel('Y Position (meters)')
    ax1.set_title('Trajectory Comparison')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axis('equal')
    
    # Plot 2: Time series of positions
    ax2.plot(time_s, raw_x, 'r:', alpha=0.7, label='Raw X')
    ax2.plot(time_s, raw_y, 'g:', alpha=0.7, label='Raw Y')
    ax2.plot(time_s, trace_x, 'b-', linewidth=2, label='Filtered X')
    ax2.plot(time_s, trace_y, 'orange', linewidth=2, label='Filtered Y')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Position (meters)')
    ax2.set_title('Position vs Time')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_velocity_analysis(df_smoothed: pl.DataFrame):
    """
    Plot velocity estimates from the Kalman filter.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    time_s = df_smoothed.get_column("time_s").to_numpy()
    vx = df_smoothed.get_column("trace_vx").to_numpy()
    vy = df_smoothed.get_column("trace_vy").to_numpy()
    
    # Calculate speed and heading
    speed = np.sqrt(vx**2 + vy**2)
    heading = np.arctan2(vy, vx) * 180 / np.pi  # Convert to degrees
    
    # Plot velocities
    ax1.plot(time_s, vx, 'b-', label='Velocity X', linewidth=2)
    ax1.plot(time_s, vy, 'r-', label='Velocity Y', linewidth=2)
    ax1.plot(time_s, speed, 'g-', label='Speed', linewidth=2)
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Velocity (m/s)')
    ax1.set_title('Velocity Components and Speed')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot heading
    ax2.plot(time_s, heading, 'purple', linewidth=2)
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Heading (degrees)')
    ax2.set_title('Vehicle Heading')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_error_analysis(df_smoothed: pl.DataFrame):
    """
    Plot the difference between raw GPS and filtered positions.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    raw_x = df_smoothed.get_column("raw_x").to_numpy()
    raw_y = df_smoothed.get_column("raw_y").to_numpy()
    trace_x = df_smoothed.get_column("trace_x").to_numpy()
    trace_y = df_smoothed.get_column("trace_y").to_numpy()
    time_s = df_smoothed.get_column("time_s").to_numpy()
    
    # Calculate errors
    error_x = raw_x - trace_x
    error_y = raw_y - trace_y
    error_magnitude = np.sqrt(error_x**2 + error_y**2)
    
    # Plot error time series
    ax1.plot(time_s, error_x, 'r-', label='X Error', alpha=0.7)
    ax1.plot(time_s, error_y, 'b-', label='Y Error', alpha=0.7)
    ax1.plot(time_s, error_magnitude, 'g-', label='Total Error', linewidth=2)
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Error (meters)')
    ax1.set_title('GPS vs Filtered Position Errors')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot error histogram
    ax2.hist(error_magnitude, bins=30, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Error Magnitude (meters)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Error Distribution')
    ax2.grid(True, alpha=0.3)
    
    # Add statistics
    mean_error = np.mean(error_magnitude)
    std_error = np.std(error_magnitude)
    ax2.axvline(mean_error, color='red', linestyle='--', 
                label=f'Mean: {mean_error:.2f}m')
    ax2.axvline(mean_error + std_error, color='orange', linestyle='--', 
                label=f'Mean + Std: {mean_error + std_error:.2f}m')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

def create_interactive_plot(df_smoothed: pl.DataFrame):
    """
    Create an interactive plot using plotly (if available).
    """
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        
        raw_x = df_smoothed.get_column("raw_x").to_numpy()
        raw_y = df_smoothed.get_column("raw_y").to_numpy()
        trace_x = df_smoothed.get_column("trace_x").to_numpy()
        trace_y = df_smoothed.get_column("trace_y").to_numpy()
        time_s = df_smoothed.get_column("time_s").to_numpy()
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Trajectory', 'X Position vs Time', 
                          'Y Position vs Time', 'Speed vs Time'),
            specs=[[{"type": "scatter"}, {"type": "scatter"}],
                   [{"type": "scatter"}, {"type": "scatter"}]]
        )
        
        # Trajectory plot
        fig.add_trace(
            go.Scatter(x=raw_x, y=raw_y, mode='markers', 
                      name='Raw GPS', marker=dict(color='red', size=4)),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=trace_x, y=trace_y, mode='lines', 
                      name='Kalman Filtered', line=dict(color='blue', width=3)),
            row=1, col=1
        )
        
        # X position vs time
        fig.add_trace(
            go.Scatter(x=time_s, y=raw_x, mode='lines', 
                      name='Raw X', line=dict(color='red', dash='dot')),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=time_s, y=trace_x, mode='lines', 
                      name='Filtered X', line=dict(color='blue')),
            row=1, col=2
        )
        
        # Y position vs time
        fig.add_trace(
            go.Scatter(x=time_s, y=raw_y, mode='lines', 
                      name='Raw Y', line=dict(color='green', dash='dot')),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(x=time_s, y=trace_y, mode='lines', 
                      name='Filtered Y', line=dict(color='orange')),
            row=2, col=1
        )
        
        # Speed vs time
        vx = df_smoothed.get_column("trace_vx").to_numpy()
        vy = df_smoothed.get_column("trace_vy").to_numpy()
        speed = np.sqrt(vx**2 + vy**2)
        
        fig.add_trace(
            go.Scatter(x=time_s, y=speed, mode='lines', 
                      name='Speed', line=dict(color='purple')),
            row=2, col=2
        )
        
        fig.update_layout(height=800, showlegend=True, 
                         title_text="Kalman Filter Results - Interactive View")
        fig.show()
        
    except ImportError:
        print("Plotly not available. Install with: pip install plotly")


# Basic plots
plot_kalman_trace(df_smoothed)
plot_velocity_analysis(df_smoothed)
plot_error_analysis(df_smoothed)

# Interactive plot (requires plotly)
create_interactive_plot(df_smoothed)

print("Visualization functions ready!")
print("Usage:")
print("1. plot_kalman_trace(df_smoothed) - Basic comparison plot")
print("2. plot_velocity_analysis(df_smoothed) - Velocity and heading")
print("3. plot_error_analysis(df_smoothed) - Error analysis")
print("4. create_interactive_plot(df_smoothed) - Interactive plotly visualization")

In [ ]:
import polars as pl
import geopandas as gpd
from shapely.geometry import Point, shape
import json

def add_sector_labels(df, geojson_path):
    # Load the sectors GeoJSON
    with open(geojson_path, 'r') as f:
        sectors_data = json.load(f)
    
    # Create list of sector polygons with their names
    sectors = []
    for feature in sectors_data['features']:
        name = feature['properties']['Name']
        geometry = shape(feature['geometry'])  # Convert to shapely geometry
        sectors.append((name, geometry))
    
    # Create a function to find the sector for each point
    def find_sector(lon, lat):
        point = Point(lon, lat)
        for name, polygon in sectors:
            if polygon.contains(point):
                # Extract just the number from "Sector X"
                return int(name.split()[1]) if "Sector" in name else 0
        return 0  # No Sector
    
    # Extract coordinates
    temp_df = {
        'locationLongitude': df['locationLongitude'].to_list(),
        'locationLatitude': df['locationLatitude'].to_list()
    }
    
    # Calculate raw sectors for all points
    raw_sectors = [find_sector(lon, lat) for lon, lat in zip(
        temp_df['locationLongitude'], 
        temp_df['locationLatitude']
    )]
    
    # Post-process: handle sequential values and carry previous sector value
    processed_sectors = []
    prev_sector = 0
    
    for i, sector in enumerate(raw_sectors):
        # If current point is in a sector, use that
        if sector != 0:
            processed_sectors.append(sector)
            prev_sector = sector
        # If current point is not in a sector (0)
        else:
            # Look ahead to see if we quickly return to a sector
            look_ahead = 5  # How many points to look ahead
            future_sectors = raw_sectors[i+1:i+look_ahead+1] if i+1 < len(raw_sectors) else []
            
            # If we quickly return to the same sector, use previous sector
            if future_sectors and any(s == prev_sector for s in future_sectors):
                next_valid_idx = next((j for j, s in enumerate(future_sectors) if s != 0), None)
                if next_valid_idx is not None and next_valid_idx <= 2:  # Only if we return within 2 points
                    processed_sectors.append(prev_sector)
                else:
                    processed_sectors.append(0)  # Use 0 for longer gaps
            else:
                processed_sectors.append(0)
    
    # Ensure wrapping from 3 to 0 (if needed)
    final_sectors = []
    for i in range(len(processed_sectors)):
        if i > 0 and processed_sectors[i-1] == 3 and processed_sectors[i] == 0:
            # Check if this is just a brief dropout or a genuine wrap
            look_ahead = min(5, len(processed_sectors) - i)
            if any(s > 0 for s in processed_sectors[i:i+look_ahead]):
                final_sectors.append(processed_sectors[i-1])  # Keep as 3
            else:
                final_sectors.append(0)  # Genuine wrap to 0
        else:
            final_sectors.append(processed_sectors[i])
    
    # Create a new dataframe with the added sector column
    return df.with_columns([
        pl.Series(name="sector", values=final_sectors)
    ])

# Example usage
df_sectors = add_sector_labels(df, 'sectors.geojson')
df_sectors.write_csv('kart_with_sectors.csv')

# Show the results
if "sector" in df_sectors.columns:
    # Print a summary showing sequences of sectors
    sectors_list = df_sectors["sector"].to_list()
    print("Sector sequence pattern:")
    current = sectors_list[0]
    count = 1
    pattern = []
    for s in sectors_list[1:]:
        if s == current:
            count += 1
        else:
            pattern.append(f"{current}×{count}")
            current = s
            count = 1
    pattern.append(f"{current}×{count}")
    print(", ".join(pattern))
    
    print("\nFirst few rows:")
    print(df_sectors.select(["locationLongitude", "locationLatitude", "sector"]).head())
else:
    print("Sector column was not created properly")

## Session 

Calculate the session based on a start finish line for a flying lap

In [ ]:
import polars as pl

# Check the data type and time range in the DataFrame
print(f"loggingTime column type: {df.schema['loggingTime']}")
print(f"Total records: {df.shape[0]}")

# Get the minimum and maximum timestamps
if df.shape[0] > 0:
    min_time = df["loggingTime"].min()
    max_time = df["loggingTime"].max()
    print(f"Data time range: {min_time} to {max_time}")
    
    # Show a few sample timestamps for better understanding
    print("\nSample timestamps:")
    print(df.select("loggingTime").sample(5, seed=42))
    
    # If you have other columns that might help identify the session
    # For example, if there's a 'session_id' or similar column
    if 'session_id' in df.columns:
        print("\nUnique session IDs:")
        print(df.select('session_id').unique().sort('session_id'))
    
    # Let's look at the distribution of timestamps
    # Create time bins to see where the data is concentrated
    print("\nData distribution by hour:")
    hour_counts = df.group_by(pl.col("loggingTime").dt.hour()).count().sort("loggingTime")
    print(hour_counts)
    
    # Based on the actual data range, suggest filter times
    print("\nSuggested filter times based on your data:")
    print(f"start_time = datetime({min_time.year}, {min_time.month}, {min_time.day}, " 
          f"{min_time.hour}, {min_time.minute}, {min_time.second}, tzinfo=timezone.utc)")
    print(f"end_time = datetime({max_time.year}, {max_time.month}, {max_time.day}, " 
          f"{max_time.hour}, {max_time.minute}, {max_time.second}, tzinfo=timezone.utc)")

## Sectors

Label each datapoint with the sector it falls with in.

Sectors are defined with a geojson file that I created on the geojson.io website.

In [ ]:
import polars as pl
import geopandas as gpd
from shapely.geometry import Point, shape
import json

def add_sector_labels(df, geojson_path):
    # Load the sectors GeoJSON
    with open(geojson_path, 'r') as f:
        sectors_data = json.load(f)
    
    # Debug - inspect coordinate ranges in your data
    lon_min = float(df['locationLongitude'].min())
    lon_max = float(df['locationLongitude'].max())
    lat_min = float(df['locationLatitude'].min())
    lat_max = float(df['locationLatitude'].max())
    print(f"Data coordinate ranges: Longitude [{lon_min}, {lon_max}], Latitude [{lat_min}, {lat_max}]")
    
    # Create list of sector polygons with their names
    sectors = []
    for feature in sectors_data['features']:
        name = feature['properties']['Name']
        geometry = shape(feature['geometry'])
        
        # Print bounds of each sector
        bounds = geometry.bounds
        print(f"{name} bounds: Longitude [{bounds[0]}, {bounds[2]}], Latitude [{bounds[1]}, {bounds[3]}]")
        
        # Add a larger buffer to capture points near boundaries
        buffered_geometry = geometry.buffer(0.0001)  # Increased buffer size
        sectors.append((name, geometry, buffered_geometry))
    
    print(f"Loaded {len(sectors)} sector polygons")
    
    # Convert string coordinates to float if they're not already
    df = df.with_columns([
        pl.col('locationLongitude').cast(pl.Float64),
        pl.col('locationLatitude').cast(pl.Float64)
    ])
    
    # Create a function to find the sector for each point
    def find_sector(lon, lat):
        point = Point(lon, lat)
        
        # First try exact containment
        for name, polygon, _ in sectors:
            if polygon.contains(point):
                return int(name.split()[1]) if "Sector" in name else 0
        
        # Then try with buffer
        for name, _, buffered in sectors:
            if buffered.contains(point):
                return int(name.split()[1]) if "Sector" in name else 0
                
        return 0  # No Sector
    
    # Extract coordinates
    temp_df = {
        'locationLongitude': df['locationLongitude'].to_list(),
        'locationLatitude': df['locationLatitude'].to_list()
    }
    
    # Calculate raw sectors for all points
    raw_sectors = [find_sector(lon, lat) for lon, lat in zip(
        temp_df['locationLongitude'], 
        temp_df['locationLatitude']
    )]
    
    # Count sector frequencies for debugging
    from collections import Counter
    sector_freq = Counter(raw_sectors)
    print("Raw sector assignments:", dict(sector_freq))
    
    # Post-process to create smoother sector transitions
    final_sectors = []
    prev_sector = 0
    window_size = 5
    
    for i in range(len(raw_sectors)):
        current = raw_sectors[i]
        
        # If point is in a sector, use that
        if current != 0:
            final_sectors.append(current)
            prev_sector = current
            continue
            
        # If point is not in a sector (0)
        # Look at a window of points before and after
        start = max(0, i - window_size)
        end = min(len(raw_sectors), i + window_size + 1)
        window = raw_sectors[start:i] + raw_sectors[i+1:end]
        
        # Filter out zeros
        window = [s for s in window if s != 0]
        
        if not window:
            # If no non-zero sectors in window, use 0
            final_sectors.append(0)
        else:
            # Count occurrences of each sector in the window
            from collections import Counter
            counts = Counter(window)
            most_common = counts.most_common(1)[0][0]
            
            # Use most common nearby sector
            final_sectors.append(most_common)
            prev_sector = most_common
    
    # Create a new dataframe with the added sector column
    return df.with_columns([
        pl.Series(name="sector", values=final_sectors)
    ])

# Example usage
df_sectors = add_sector_labels(df, 'sectors.geojson')
df_sectors.write_csv('kart_with_sectors.csv')

# Show the results
if "sector" in df_sectors.columns:
    print(df_sectors.select(["locationLongitude", "locationLatitude", "sector"]).head())
    
    # Count how many points are in each sector - using Polars syntax
    sector_counts = df_sectors.filter(pl.col("sector") == 0).shape[0]
    print(f"\nPoints in sector 0: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 1).shape[0]
    print(f"Points in sector 1: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 2).shape[0]
    print(f"Points in sector 2: {sector_counts}")
    sector_counts = df_sectors.filter(pl.col("sector") == 3).shape[0]
    print(f"Points in sector 3: {sector_counts}")

In [ ]:
df_labeled.write_csv('kart_race_data_sectors.json')


## G-force

Calculate G-Force. Since the orientation of the phane changes, need to calculate it.

In [ ]:
import polars as pl
import numpy as np

def create_calculated_gforce_df(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates a new DataFrame with only the calculated total_g_force column.
    """
    # First, calculate the new column on the existing dataframe
    df_with_gforce = df.with_columns(
        (
            pl.col("motionUserAccelerationX").cast(pl.Float64)**2 +
            pl.col("motionUserAccelerationY").cast(pl.Float64)**2 +
            pl.col("motionUserAccelerationZ").cast(pl.Float64)**2
        ).sqrt().alias("total_g_force")
    )
    
    # Then, select only the new column from the new dataframe
    return df_with_gforce.select(pl.col("total_g_force"))


## Time since start

This function calculates a cumulative time_since_start column, which is the basis for all lap and sector timing analysis.

In [ ]:
import polars as pl

def calculate_time_since_start(df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates time in seconds since the start of the recording.
    """
    start_time = df.select(pl.col("loggingTime").min()).item()
    return df.with_columns(
        (pl.col("loggingTime") - start_time).alias("time_since_start")
    )

## Delta Time

This function calculates the time difference between your current lap and a reference (best) lap, allowing you to see if you are gaining or losing time. You'll need to define the best lap data separately.

In [ ]:
import polars as pl

def calculate_delta_time(df: pl.DataFrame, best_lap_df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates the time difference (delta) compared to a best lap.
    Requires aligning data based on distance or time.
    """
    # For a simple example, let's assume we align on a 'distance' column
    # You would typically resample or join to align the data points
    return df.join(best_lap_df, on="pedometerDistance", suffix="_best", how="left").with_columns(
        (pl.col("time_since_start") - pl.col("time_since_start_best")).alias("delta_time")
    )


## Speed and Velocity

This function calculates a smoothed speed and velocity from the location data to reduce GPS jitter.

In [ ]:
import polars as pl
import numpy as np

def calculate_smoothed_speed(df: pl.DataFrame, window_size: int = 5) -> pl.DataFrame:
    """
    Calculates a rolling average of location speed to smooth out data.
    """
    return df.with_columns(
        pl.col("locationSpeed").rolling_mean(window_size=window_size).alias("smoothed_speed")
    )


In [ ]:
df_with_gforce = create_calculated_gforce_df(df)


In [ ]:
df_with_gforce.head()